In [ ]:
# === COLAB SETUP (VSCode-friendly, no drive.mount required) ===
# VSCode + Colab + drive.mount() is broken — the OAuth popup has nowhere to
# surface. We bypass Drive entirely and pull the zip via gdown using a public
# Drive share link.
#
# IMPORTANT: in Drive, the file at FILE_ID below MUST be shared as
# "Anyone with the link" → Viewer, BEFORE running this cell. If it's
# restricted, gdown gets an HTML interstitial back instead of the zip and
# the extract step will fail.

import os, zipfile, time, glob, subprocess
from pathlib import Path

# Pre-filled with the user's Drive share-link FILE_ID:
# https://drive.google.com/file/d/1mCBWNyjP4XsciDzC_Hwm8phndyWIYSxC/view
FILE_ID = "1mCBWNyjP4XsciDzC_Hwm8phndyWIYSxC"

ZIP_LOCAL = Path("/content/multimodal-cancer-classification-challenge-2026.zip")
LOCAL_DATA_PARENT = Path("/content/data")
LOCAL_DATA_PARENT.mkdir(parents=True, exist_ok=True)

# Outputs go to LOCAL /content/v16_runs/ — wiped when the runtime ends.
# At the end of training, download from VSCode's file explorer (right-click
# the folder → Download) or copy to Drive via a separate browser Colab tab.
COLAB_OUT_DIR = Path("/content/v16_runs")
COLAB_OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Download the zip (one-time per runtime) ---
if not ZIP_LOCAL.exists():
    print("Installing gdown ...")
    subprocess.run(["pip", "install", "-q", "gdown"], check=True)
    print(f"Downloading zip (FILE_ID={FILE_ID}) ...")
    t0 = time.time()
    # --fuzzy accepts the full Drive URL and handles the large-file confirm step.
    url = f"https://drive.google.com/uc?id={FILE_ID}"
    r = subprocess.run(["gdown", "--fuzzy", url, "-O", str(ZIP_LOCAL)])
    if r.returncode != 0 or not ZIP_LOCAL.exists() or ZIP_LOCAL.stat().st_size < 1_000_000:
        actual = ZIP_LOCAL.stat().st_size if ZIP_LOCAL.exists() else "(not found)"
        raise RuntimeError(
            f"gdown failed (returncode={r.returncode}, file size={actual}). "
            f"Most common cause: the Drive file at FILE_ID={FILE_ID} is not "
            f"set to 'Anyone with the link'. Fix in Drive: right-click the "
            f"zip → Share → General access → 'Anyone with the link' → Viewer.")
    print(f"  downloaded in {time.time()-t0:.1f}s, size={ZIP_LOCAL.stat().st_size:,} bytes")
else:
    print(f"Zip already present at {ZIP_LOCAL} ({ZIP_LOCAL.stat().st_size:,} bytes)")

# --- Extract (one-time per runtime) ---
if not list(LOCAL_DATA_PARENT.glob("**/train.csv")):
    print(f"Extracting...")
    t0 = time.time()
    with zipfile.ZipFile(ZIP_LOCAL) as z:
        z.extractall(LOCAL_DATA_PARENT)
    print(f"  done in {time.time()-t0:.1f}s")
else:
    print(f"Data already extracted under {LOCAL_DATA_PARENT}")

# --- Find the directory that contains train.csv ---
candidates = sorted(glob.glob(str(LOCAL_DATA_PARENT / "**" / "train.csv"), recursive=True))
assert candidates, f"train.csv not found under {LOCAL_DATA_PARENT}"
COLAB_DATA_ROOT = Path(candidates[0]).parent

print(f"\nCOLAB_DATA_ROOT = {COLAB_DATA_ROOT}")
print(f"COLAB_OUT_DIR   = {COLAB_OUT_DIR}  (LOCAL, EPHEMERAL — copy out before runtime ends!)")
print(f"\nContents of COLAB_DATA_ROOT:")
for p in sorted(COLAB_DATA_ROOT.iterdir()):
    kind = "dir" if p.is_dir() else f"{p.stat().st_size:,} bytes"
    print(f"  {p.name}  ({kind})")

# Multimodal Cancer Classification Challenge 2026 — v16 (Colab port)

**This is the Colab port of `improvedv16_source.ipynb`.** Logic, hyperparameters,
and model code are bit-identical. Only the paths differ:

- `DATA_ROOT` is set by the `colab-setup` cell to a fast local SSD path under
  `/content/data/` — the competition zip is extracted from Drive once per
  Colab session.
- `OUT_DIR` points to `/content/drive/.../v16_colab_runs/` on Drive so all
  ckpts, history JSONs, submission CSVs, and `learning_curves.png` survive
  Colab disconnects. Restart the notebook and the skip-on-existing logic
  resumes from the last completed fold/seed.

After the run finishes, download `submission.csv` from
`/content/drive/.../v16_colab_runs/` and upload it to the Kaggle competition
page (no Kaggle commit needed — saves the GPU quota for another iteration).

---

**Premise (from v15 code review):** Our 3-fold CV is lying to us. v15 had the best CV mean (0.866) of any version, the worst LB (0.572). Two folds matched train-distribution easily, one fold was hard, and the average buried the failure mode. Worse, mixup was fighting the CoMIR features — the contrastive backbone learned BF↔FL alignment, and mixup trained the model on synthetic *mixtures* of two cells, which have no real cross-modal alignment, dragging the backbone away from its useful invariance.

## Nine changes from v15

1. **No mixup** during supervised fine-tune. Keeps the CoMIR alignment intact.
2. **LOPO cross-validation** (12 folds, one patient out per fold, single seed). Honest OOD estimator.
3. **Heavier stain augmentation during CoMIR SSL.** ColorJitter(0.5, 0.5) + RandomGamma γ∈[0.7, 1.4] per modality.
4. **Logit-space TTA averaging.** Sum logits across 8 D4 augmentations, sigmoid once.
5. **Ensemble = all LOPO snapshots + multi-seed full-data snapshots**, rank-averaged at submission time.
6. **Label smoothing (ε = 0.05) on BCE targets**, per L4 lecture p.48.
7. **Multi-snapshot ensembling** at epochs {3, 5} per fold.
8. **EMA of supervised weights (decay 0.999).** Saved as the snapshot ckpts.
9. **Aux SSL-alignment loss during supervised fine-tune** — frozen CoMIR projection heads + `0.05 × NT-Xent(z_bf, z_fl)`.

Plus the `submission_patient_agg.csv` hedge for patient-level scoring.

## Approximate runtime on a Colab T4 (single GPU)

Same as Kaggle T4×2 (we only use GPU 0 either way): ~8 h 15 min. Colab Free
disconnects after ~12 h of continuous runtime — so this fits, but only just.
If Colab pre-empts mid-run, restart and the snapshot-skip logic resumes from
the last completed fold/seed (drive-persisted ckpts).


In [ ]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"

import re, io, json, time, random, glob, functools, gc
from pathlib import Path

print = functools.partial(print, flush=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import models
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import LeaveOneGroupOut
from scipy.stats import rankdata
from PIL import Image
import matplotlib.pyplot as plt

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "n_gpu:", torch.cuda.device_count())
!nvidia-smi -L

In [ ]:
# Colab port: DATA_ROOT and OUT_DIR are set by the colab-setup cell above.
DATA_ROOT = COLAB_DATA_ROOT
assert (DATA_ROOT / "train.csv").exists(), f"train.csv not at {DATA_ROOT}"
print("DATA_ROOT =", DATA_ROOT)

OUT_DIR = COLAB_OUT_DIR   # on Drive — survives disconnects
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR =", OUT_DIR)

# CV: LOPO replaces 3-fold × 2 seeds. One model per held-out patient.
BASE_SEED   = 1
SUPERVISED_SEED = 7

# Supervised optimization
EPOCHS         = 6                   # fixed (no early stopping under LOPO)
BATCH_SIZE     = 128
BACKBONE_LR    = 3e-5                # 1/10 of head LR per L4 lecture
HEAD_LR        = 3e-4
WEIGHT_DECAY   = 1e-4
GRAD_CLIP      = 1.0
DROPOUT        = 0.3
PCT_START      = 0.1
LABEL_SMOOTHING = 0.05               # ε per L4 p.48; applied only during training
SNAPSHOT_EPOCHS = [3, 5]
assert all(0 <= e < EPOCHS for e in SNAPSHOT_EPOCHS), \
    "SNAPSHOT_EPOCHS must be inside [0, EPOCHS)"

# EMA decay tuned for our actual training length (~426 steps/fold).
# Conventional 0.999 has an effective averaging window of 1/(1-decay)=1000 steps,
# which is larger than the entire training run — the shadow would stay ~65%
# biased toward initial weights at snapshot time, making EMA an effective no-op
# (or worse, polluting the snapshot with pre-trained junk).
# 0.99 → ~100-step window (~1.4 epochs), which actually smooths the OneCycleLR
# cooldown phase as intended.
EMA_DECAY = 0.99
AUX_ALIGN_WEIGHT = 0.05
FULL_DATA_SEEDS = [107, 207]
# NOTE: mixup is intentionally DISABLED in v16 — see code review (item #1)

# CoMIR-style contrastive SSL
SSL_ENABLED      = True
SSL_EPOCHS       = 5
SSL_LR           = 1e-3
SSL_BATCH        = 256
SSL_TEMPERATURE  = 0.1
SSL_PROJ_DIM     = 128
SSL_CKPT         = OUT_DIR / "ssl_comir_backbone.pt"
SSL_COLOR_BC     = 0.5               # ColorJitter brightness=contrast=this
SSL_GAMMA_RANGE  = (0.7, 1.4)

# Mild aug for supervised fine-tune
TRAIN_COLOR_BC   = 0.2

# Sampler
NUM_WORKERS = 2
PATIENTS_PER_BATCH = 4

# Full-data model trained on all 12 patients for the same EPOCHS
TRAIN_FULL_DATA_MODEL = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.cuda.set_device(0)

# Dataset-level normalization (v11/v13/v15 stats)
BF_MEAN, BF_STD = 0.504, 0.216
FL_MEAN, FL_STD = 0.100, 0.144

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def worker_init_fn(worker_id):
    s = torch.initial_seed() % (2**32)
    np.random.seed(s); random.seed(s)

seed_everything(BASE_SEED)


In [ ]:
PAT_RE = re.compile(r"^pat_(\d+)_image_\d+\.jpg$")

def parse_patient_id(filename):
    m = PAT_RE.match(Path(filename).name)
    return int(m.group(1)) if m else None

def load_train_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    df["patient_id"] = df["Name"].map(parse_patient_id).astype(int)
    return df

def load_test_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    return df

def cache_split(names, bf_dir, fl_dir, label=""):
    bf_dir, fl_dir = Path(bf_dir), Path(fl_dir)
    bf, fl = {}, {}
    t0 = time.time()
    for i, n in enumerate(names):
        with open(bf_dir / n, "rb") as f: bf[n] = f.read()
        with open(fl_dir / n, "rb") as f: fl[n] = f.read()
        if (i + 1) % 20000 == 0:
            print(f"  [{label}] cached {i+1}/{len(names)} in {time.time()-t0:.1f}s")
    print(f"  [{label}] cached {len(names)} in {time.time()-t0:.1f}s")
    return bf, fl

class CachedCellDataset(Dataset):
    def __init__(self, df, bf_cache, fl_cache, bf_tf, fl_tf, paired_tf=None):
        self.df = df.reset_index(drop=True)
        self.bf_cache = bf_cache; self.fl_cache = fl_cache
        self.bf_tf = bf_tf; self.fl_tf = fl_tf
        self.paired_tf = paired_tf
    def __len__(self): return len(self.df)
    @staticmethod
    def _decode(buf): return Image.open(io.BytesIO(buf)).convert("L")
    def __getitem__(self, idx):
        row = self.df.iloc[idx]; name = row["Name"]
        bf = self.bf_tf(self._decode(self.bf_cache[name]))
        fl = self.fl_tf(self._decode(self.fl_cache[name]))
        if self.paired_tf is not None:
            bf, fl = self.paired_tf(bf, fl)
        label = int(row["Diagnosis"]) if "Diagnosis" in row else -1
        return {"bf": bf, "fl": fl, "label": label, "name": name}

class ContrastivePairDataset(Dataset):
    def __init__(self, all_pairs, bf_caches, fl_caches, bf_tf, fl_tf, paired_tf):
        self.pairs = all_pairs
        self.bf_caches = bf_caches
        self.fl_caches = fl_caches
        self.bf_tf = bf_tf; self.fl_tf = fl_tf
        self.paired_tf = paired_tf
    def __len__(self): return len(self.pairs)
    @staticmethod
    def _decode(buf): return Image.open(io.BytesIO(buf)).convert("L")
    def __getitem__(self, idx):
        name, split = self.pairs[idx]
        bf = self.bf_tf(self._decode(self.bf_caches[split][name]))
        fl = self.fl_tf(self._decode(self.fl_caches[split][name]))
        if self.paired_tf is not None:
            bf, fl = self.paired_tf(bf, fl)
        return {"bf": bf, "fl": fl}

In [ ]:
def leave_one_patient_out_splits(df):
    """12-fold LOPO. Each fold holds out exactly one patient."""
    logo = LeaveOneGroupOut()
    return list(logo.split(df, groups=df["patient_id"].to_numpy()))

def summarize_lopo(df, tr, va):
    trd, vad = df.iloc[tr], df.iloc[va]
    held_out = sorted(vad["patient_id"].unique().tolist())
    held_label = vad["Diagnosis"].iloc[0]
    return (f"train: {len(tr):>6} cells, {trd['patient_id'].nunique():>2}p, "
            f"pos {trd['Diagnosis'].mean():.3f} | "
            f"val (held-out): pat {held_out[0]}, n={len(va)}, label={held_label}")

class PatientBalancedSampler(Sampler):
    """Each mini-batch contains cells from `patients_per_batch` distinct patients."""
    def __init__(self, df, batch_size, patients_per_batch=4, seed=0):
        assert batch_size % patients_per_batch == 0
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.per_pat = batch_size // patients_per_batch
        self.patients_per_batch = patients_per_batch
        self.rng = np.random.default_rng(seed)
        self.by_pat = {p: np.array(g.index.tolist())
                       for p, g in self.df.groupby("patient_id")}
        self.patients = list(self.by_pat.keys())
        self.epoch_len = len(self.df) // batch_size * batch_size
    def __len__(self): return self.epoch_len
    def __iter__(self):
        out = []
        for _ in range(self.epoch_len // self.batch_size):
            pats = self.rng.choice(self.patients,
                                   size=min(self.patients_per_batch, len(self.patients)),
                                   replace=False)
            for p in pats:
                idxs = self.by_pat[p]
                out.extend(self.rng.choice(idxs, size=self.per_pat,
                                           replace=len(idxs) < self.per_pat).tolist())
        return iter(out)

def _to_tensor_norm(mean, std):
    def fn(img):
        t = TF.to_tensor(img)
        return TF.normalize(t, [mean], [std])
    return fn
to_tensor_bf = _to_tensor_norm(BF_MEAN, BF_STD)
to_tensor_fl = _to_tensor_norm(FL_MEAN, FL_STD)

class RandomGamma:
    """Apply random gamma correction to a PIL image (before ToTensor)."""
    def __init__(self, lo=0.7, hi=1.4, p=0.8):
        self.lo, self.hi, self.p = lo, hi, p
    def __call__(self, img):
        if random.random() < self.p:
            g = random.uniform(self.lo, self.hi)
            img = TF.adjust_gamma(img, gamma=g)
        return img

class PairedGeoAug:
    """Identical geometric aug applied to BF and FL so cross-modal alignment is preserved."""
    def __init__(self, p_hflip=0.5, p_vflip=0.5, rot90=True, max_rot=10.0):
        self.p_hflip = p_hflip; self.p_vflip = p_vflip
        self.rot90 = rot90; self.max_rot = max_rot
    def __call__(self, bf, fl):
        if self.rot90:
            k = random.randint(0, 3)
            if k:
                bf = torch.rot90(bf, k, dims=(-2, -1))
                fl = torch.rot90(fl, k, dims=(-2, -1))
        if random.random() < self.p_hflip: bf, fl = TF.hflip(bf), TF.hflip(fl)
        if random.random() < self.p_vflip: bf, fl = TF.vflip(bf), TF.vflip(fl)
        if self.max_rot > 0:
            a = random.uniform(-self.max_rot, self.max_rot)
            bf, fl = TF.rotate(bf, a), TF.rotate(fl, a)
        return bf, fl

def train_modality_transform(modality):
    """Mild stain aug for supervised fine-tune (keeps CoMIR features stable)."""
    norm = to_tensor_bf if modality == "bf" else to_tensor_fl
    return T.Compose([T.ColorJitter(brightness=TRAIN_COLOR_BC, contrast=TRAIN_COLOR_BC), norm])

def eval_modality_transform(modality):
    return to_tensor_bf if modality == "bf" else to_tensor_fl

def ssl_modality_transform(modality):
    """HEAVY stain aug for CoMIR — forces invariance across simulated staining batches.
    Per-modality, independent randomness so each batch shows BF and FL drawn from
    different points on the stain manifold. The alignment task has to learn the
    underlying *content* correspondence, not surface stain coupling."""
    norm = to_tensor_bf if modality == "bf" else to_tensor_fl
    return T.Compose([
        T.ColorJitter(brightness=SSL_COLOR_BC, contrast=SSL_COLOR_BC),
        RandomGamma(*SSL_GAMMA_RANGE, p=0.8),
        norm,
    ])

In [ ]:
def _make_resnet18_branch(pretrained=True):
    weights = "DEFAULT" if pretrained else None
    net = models.resnet18(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features; net.fc = nn.Identity()
    return net, fd  # 512

def _make_proj_head(fd, proj_dim=SSL_PROJ_DIM):
    """Same projection architecture in SSL and (frozen) in supervised aux loss.
    Two-layer MLP fd→fd→proj_dim, matching SimCLR/CoMIR."""
    return nn.Sequential(
        nn.Linear(fd, fd), nn.ReLU(inplace=True), nn.Linear(fd, proj_dim))

class MultimodalClassifier(nn.Module):
    """v16: now carries the SSL projection heads alongside the classifier head.
    The projections are loaded from the SSL ckpt and FROZEN; they're used only
    by `forward_with_proj` for the aux NT-Xent alignment loss during training.
    At inference time, only `forward` is called — projections cost nothing."""
    def __init__(self, pretrained=True, dropout=DROPOUT):
        super().__init__()
        self.bf_branch, fd = _make_resnet18_branch(pretrained)
        self.fl_branch, _  = _make_resnet18_branch(pretrained)
        self.bf_proj = _make_proj_head(fd)
        self.fl_proj = _make_proj_head(fd)
        self.head = nn.Sequential(
            nn.Linear(fd * 2, 256), nn.BatchNorm1d(256),
            nn.ReLU(inplace=True), nn.Dropout(dropout), nn.Linear(256, 1))
    def forward(self, bf, fl):
        feat = torch.cat([self.bf_branch(bf), self.fl_branch(fl)], dim=1)
        return self.head(feat).squeeze(-1)
    def forward_with_proj(self, bf, fl):
        """Return logits + L2-normalized projections for the aux NT-Xent loss."""
        bf_feat = self.bf_branch(bf)
        fl_feat = self.fl_branch(fl)
        z_bf = F.normalize(self.bf_proj(bf_feat), dim=1)
        z_fl = F.normalize(self.fl_proj(fl_feat), dim=1)
        feat = torch.cat([bf_feat, fl_feat], dim=1)
        logits = self.head(feat).squeeze(-1)
        return logits, z_bf, z_fl

class ContrastiveModel(nn.Module):
    """SSL model — same backbones + projection heads as MultimodalClassifier.
    After SSL, all four state dicts (branches + projections) are saved to
    SSL_CKPT and loaded into MultimodalClassifier for supervised."""
    def __init__(self, pretrained=True, proj_dim=SSL_PROJ_DIM):
        super().__init__()
        self.bf_branch, fd = _make_resnet18_branch(pretrained)
        self.fl_branch, _  = _make_resnet18_branch(pretrained)
        self.bf_proj = _make_proj_head(fd, proj_dim)
        self.fl_proj = _make_proj_head(fd, proj_dim)
    def forward(self, bf, fl):
        z_bf = F.normalize(self.bf_proj(self.bf_branch(bf)), dim=1)
        z_fl = F.normalize(self.fl_proj(self.fl_branch(fl)), dim=1)
        return z_bf, z_fl

def nt_xent_loss(z_bf, z_fl, tau=SSL_TEMPERATURE):
    B = z_bf.size(0)
    z = torch.cat([z_bf, z_fl], dim=0)
    sim = z @ z.t() / tau
    eye = torch.eye(2 * B, dtype=torch.bool, device=z.device)
    sim.masked_fill_(eye, float("-inf"))
    targets = (torch.arange(2 * B, device=z.device) + B) % (2 * B)
    return F.cross_entropy(sim, targets)

with torch.no_grad():
    _m = MultimodalClassifier(pretrained=False).cpu()
    _x = torch.zeros(2, 1, 128, 128)
    print("Classifier output shape:", _m(_x, _x).shape,
          "  params:", sum(p.numel() for p in _m.parameters()) // 1_000_000, "M")
    _logits, _z_bf, _z_fl = _m.forward_with_proj(_x, _x)
    print("forward_with_proj shapes:", _logits.shape, _z_bf.shape, _z_fl.shape)
    _c = ContrastiveModel(pretrained=False).cpu()
    z_bf, z_fl = _c(_x, _x)
    print("Contrastive output shapes:", z_bf.shape, z_fl.shape,
          " NT-Xent on tiny batch:", float(nt_xent_loss(z_bf, z_fl)))
    del _m, _c, _x, _logits, _z_bf, _z_fl, z_bf, z_fl

In [ ]:
df_train = load_train_df(DATA_ROOT / "train.csv")
df_test  = load_test_df(DATA_ROOT / "sampleSubmission.csv")
print(f"Train: {len(df_train)} cells, {df_train['patient_id'].nunique()} patients, "
      f"pos rate {df_train['Diagnosis'].mean():.4f}")
print(f"Test:  {len(df_test)} cells")

print("\nPer-patient distribution (train):")
pp = df_train.groupby("patient_id").agg(
    n_cells=("Name", "size"), label=("Diagnosis", "first")).reset_index()
print(pp.to_string(index=False))

print("\n--- Caching JPEG bytes into RAM ---")
bf_train_cache, fl_train_cache = cache_split(
    df_train["Name"].tolist(),
    DATA_ROOT / "BF" / "train", DATA_ROOT / "FL" / "train", label="train")
bf_test_cache, fl_test_cache = cache_split(
    df_test["Name"].tolist(),
    DATA_ROOT / "BF" / "test", DATA_ROOT / "FL" / "test", label="test")
approx_mb = (sum(len(b) for b in bf_train_cache.values()) +
             sum(len(b) for b in fl_train_cache.values()) +
             sum(len(b) for b in bf_test_cache.values()) +
             sum(len(b) for b in fl_test_cache.values())) / (1024 * 1024)
print(f"\nApprox RAM used by JPEG cache: {approx_mb:.0f} MB")

In [ ]:
def train_contrastive_ssl(epochs=SSL_EPOCHS, batch_size=SSL_BATCH, lr=SSL_LR):
    seed_everything(BASE_SEED + 7)
    all_pairs = ([(n, "train") for n in df_train["Name"].tolist()] +
                 [(n, "test")  for n in df_test["Name"].tolist()])
    print(f"SSL on {len(all_pairs)} BF↔FL pairs (train + test combined)")
    print(f"   stain aug: ColorJitter({SSL_COLOR_BC}) + RandomGamma{SSL_GAMMA_RANGE}")
    random.shuffle(all_pairs)
    bf_caches = {"train": bf_train_cache, "test": bf_test_cache}
    fl_caches = {"train": fl_train_cache, "test": fl_test_cache}
    ds = ContrastivePairDataset(all_pairs, bf_caches, fl_caches,
                                ssl_modality_transform("bf"),
                                ssl_modality_transform("fl"),
                                paired_tf=PairedGeoAug())
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                        worker_init_fn=worker_init_fn)

    model = ContrastiveModel(pretrained=True).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, steps_per_epoch=len(loader),
        epochs=epochs, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None

    for ep in range(epochs):
        t0 = time.time(); losses = []
        model.train()
        for i, batch in enumerate(loader):
            bf = batch["bf"].to(DEVICE, non_blocking=True)
            fl = batch["fl"].to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=scaler is not None):
                z_bf, z_fl = model(bf, fl)
                loss = nt_xent_loss(z_bf, z_fl)
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                old_scale = scaler.get_scale(); scaler.step(optimizer); scaler.update()
                if scaler.get_scale() >= old_scale: sched.step()
            else:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); sched.step()
            losses.append(loss.item())
            if (i + 1) % 100 == 0:
                print(f"    step {i+1}/{len(loader)} | loss {float(np.mean(losses[-100:])):.4f}")
        print(f"  CoMIR ep {ep} | loss {float(np.mean(losses)):.4f} | {time.time()-t0:.1f}s")

    # Save all four state dicts. v16 supervised stage uses the projections
    # (frozen) for the aux NT-Xent alignment loss.
    torch.save({"bf_branch": model.bf_branch.state_dict(),
                "fl_branch": model.fl_branch.state_dict(),
                "bf_proj":   model.bf_proj.state_dict(),
                "fl_proj":   model.fl_proj.state_dict()}, SSL_CKPT)
    print(f"Saved CoMIR backbone + projection heads to {SSL_CKPT}")
    del model, optimizer, sched, scaler, loader, ds
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

if SSL_ENABLED and not SSL_CKPT.exists():
    print("=== Stage 1: Contrastive SSL (CoMIR + heavy stain aug) ===")
    train_contrastive_ssl()
elif SSL_CKPT.exists():
    print(f"CoMIR backbone exists at {SSL_CKPT} — skipping pretraining.")
else:
    print("(SSL disabled — will use ImageNet init)")

In [ ]:
def load_ssl_branches(model, ssl_ckpt_path=SSL_CKPT):
    """Load CoMIR backbones + (when present) projection heads into a
    MultimodalClassifier. Projections are required when AUX_ALIGN_WEIGHT>0."""
    if not Path(ssl_ckpt_path).exists():
        print(f"  (no SSL ckpt at {ssl_ckpt_path}, keeping ImageNet init)"); return
    state = torch.load(ssl_ckpt_path, map_location="cpu", weights_only=False)
    msg_bf = model.bf_branch.load_state_dict(state["bf_branch"], strict=False)
    msg_fl = model.fl_branch.load_state_dict(state["fl_branch"], strict=False)
    if "bf_proj" in state and "fl_proj" in state:
        model.bf_proj.load_state_dict(state["bf_proj"], strict=False)
        model.fl_proj.load_state_dict(state["fl_proj"], strict=False)
        proj_loaded = True
    else:
        proj_loaded = False
        if AUX_ALIGN_WEIGHT > 0.0:
            raise RuntimeError(
                f"SSL ckpt at {ssl_ckpt_path} has no projection heads but "
                f"AUX_ALIGN_WEIGHT={AUX_ALIGN_WEIGHT}>0 needs them. Delete the "
                f"stale ckpt to force retraining, or set AUX_ALIGN_WEIGHT=0.0.")
    print(f"  loaded CoMIR backbone (bf missing={len(msg_bf.missing_keys)}, "
          f"fl missing={len(msg_fl.missing_keys)}, proj_loaded={proj_loaded})")

def make_discriminative_optimizer(model, backbone_lr=BACKBONE_LR, head_lr=HEAD_LR,
                                  weight_decay=WEIGHT_DECAY):
    """Two param groups (backbone, head). Projection heads are FROZEN so the
    aux loss anchors the backbone to CoMIR projections (which are themselves
    fixed); the projections never drift away from their SSL solution."""
    backbone_params = list(model.bf_branch.parameters()) + list(model.fl_branch.parameters())
    head_params     = list(model.head.parameters())
    for p in model.bf_proj.parameters(): p.requires_grad = False
    for p in model.fl_proj.parameters(): p.requires_grad = False
    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": backbone_lr},
        {"params": head_params,     "lr": head_lr},
    ], weight_decay=weight_decay)
    return optimizer, [backbone_lr, head_lr]

class ModelEMA:
    """Exponential moving average of model state. Updates a shadow copy after
    each *successful* optimizer step (under AMP, skip if the scaler rejected
    the step — those params are stale). At snapshot time, save the shadow."""
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)
            else:
                # Integer buffers like num_batches_tracked: copy verbatim.
                self.shadow[k] = v.detach().clone()
    def state_dict(self):
        return self.shadow

def run_epoch(model, loader, optimizer, scaler, criterion, train,
              grad_clip=0.0, sched=None, log_every=0,
              label_smoothing=0.0, aux_align_weight=0.0, ema=None,
              sanity_print=False):
    """v16: BCE + (optional) aux NT-Xent on frozen SSL projections, with EMA
    shadow update after each successful step. Label smoothing applied to BCE
    target ONLY when train=True; AUC/val_loss use unsmoothed labels.
    """
    model.train(train)
    losses, hard_ys, ps, names = [], [], [], []
    t_last = time.time()
    for i, batch in enumerate(loader):
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        y  = batch["label"].float().to(DEVICE, non_blocking=True)
        hard_ys.append(batch["label"].numpy())
        if not train: names.extend(batch["name"])
        if train and label_smoothing > 0.0:
            y_target = y * (1.0 - 2.0 * label_smoothing) + label_smoothing
        else:
            y_target = y
        with torch.amp.autocast("cuda", enabled=scaler is not None):
            if train and aux_align_weight > 0.0:
                logits, z_bf, z_fl = model.forward_with_proj(bf, fl)
                bce = criterion(logits, y_target)
                aux = nt_xent_loss(z_bf, z_fl, tau=SSL_TEMPERATURE)
                loss = bce + aux_align_weight * aux
                if sanity_print and i == 0:
                    print(f"    [sanity] bce={float(bce):.4f}  "
                          f"aux={float(aux):.4f}  "
                          f"weighted_aux={aux_align_weight * float(aux):.4f}")
            else:
                logits = model(bf, fl)
                loss = criterion(logits, y_target)
        if train:
            optimizer.zero_grad(set_to_none=True)
            step_ok = False
            if scaler is not None:
                scaler.scale(loss).backward()
                if grad_clip > 0:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                old_scale = scaler.get_scale()
                scaler.step(optimizer); scaler.update()
                step_ok = (scaler.get_scale() >= old_scale)
                if sched is not None and step_ok:
                    sched.step()
            else:
                loss.backward()
                if grad_clip > 0: nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
                if sched is not None: sched.step()
                step_ok = True
            if ema is not None and step_ok:
                ema.update(model)
        losses.append(loss.item())
        ps.append(torch.sigmoid(logits).detach().float().cpu().numpy())
        if log_every and (i + 1) % log_every == 0:
            dt = time.time() - t_last; t_last = time.time()
            print(f"    step {i+1}/{len(loader)} | {dt:.1f}s | loss {float(np.mean(losses[-log_every:])):.4f}")
    hard_ys = np.concatenate(hard_ys); ps = np.concatenate(ps)
    auc = roc_auc_score(hard_ys, ps) if len(np.unique(hard_ys)) > 1 else float("nan")
    return float(np.mean(losses)), auc, hard_ys, ps, names

def train_one_lopo_fold(train_df, val_df, ckpt_pattern, oof_path, hist_path,
                         epochs=EPOCHS, seed=SUPERVISED_SEED,
                         snapshot_epochs=SNAPSHOT_EPOCHS):
    """Train on 11 patients with EMA + aux NT-Xent. Snapshot the EMA shadow
    at each epoch in `snapshot_epochs`. Save predictions on the held-out
    patient per snapshot. No epoch selection — every snapshot enters the
    final ensemble."""
    assert "{ep}" in str(ckpt_pattern), "ckpt_pattern must contain '{ep}'"
    seed_everything(seed)
    train_ds = CachedCellDataset(train_df, bf_train_cache, fl_train_cache,
                                 train_modality_transform("bf"),
                                 train_modality_transform("fl"),
                                 paired_tf=PairedGeoAug())
    sampler = PatientBalancedSampler(train_df, batch_size=BATCH_SIZE,
                                     patients_per_batch=PATIENTS_PER_BATCH, seed=seed)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                              worker_init_fn=worker_init_fn)
    val_ds = CachedCellDataset(val_df, bf_train_cache, fl_train_cache,
                               eval_modality_transform("bf"),
                               eval_modality_transform("fl"))
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True,
                            worker_init_fn=worker_init_fn)

    model = MultimodalClassifier(pretrained=True, dropout=DROPOUT).to(DEVICE)
    if SSL_ENABLED: load_ssl_branches(model)

    pos = (train_df["Diagnosis"] == 1).sum()
    neg = (train_df["Diagnosis"] == 0).sum()
    pos_weight = torch.tensor(neg / max(pos, 1), device=DEVICE)
    print(f"  pos_weight={pos_weight.item():.3f}  seed={seed}  epochs={epochs}  "
          f"backbone_lr={BACKBONE_LR}  head_lr={HEAD_LR}  "
          f"ls={LABEL_SMOOTHING}  aux={AUX_ALIGN_WEIGHT}  ema={EMA_DECAY}  "
          f"snapshots@{snapshot_epochs}  no_mixup")
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer, max_lrs = make_discriminative_optimizer(model)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=max_lrs,
        steps_per_epoch=len(train_loader), epochs=epochs, pct_start=PCT_START)
    scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None
    ema = ModelEMA(model, decay=EMA_DECAY)

    history = []
    snapshot_preds = {}
    vy_canon = vn_canon = None
    for ep in range(epochs):
        t0 = time.time()
        tr_loss, tr_auc, _, _, _ = run_epoch(
            model, train_loader, optimizer, scaler, criterion, True,
            grad_clip=GRAD_CLIP, sched=sched, log_every=300,
            label_smoothing=LABEL_SMOOTHING,
            aux_align_weight=AUX_ALIGN_WEIGHT, ema=ema,
            sanity_print=(ep == 0))

        # Swap raw -> EMA for val + snapshot; restore at end of epoch.
        raw_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(ema.state_dict())
        with torch.no_grad():
            va_loss, _va_auc, vy, vp, vn = run_epoch(
                model, val_loader, None, None, criterion, False)
        if vy_canon is None:
            vy_canon, vn_canon = vy, vn
        held_class = int(vy[0]) if len(vy) else -1
        held_mean  = float(vp.mean()) if len(vp) else float("nan")
        if ep in snapshot_epochs:
            ckpt_path = Path(str(ckpt_pattern).replace("{ep}", str(ep)))
            torch.save({"model": model.state_dict(), "epoch": ep,
                        "args": {"dropout": DROPOUT}}, ckpt_path)
            snapshot_preds[ep] = vp
            print(f"    [snapshot] saved ep{ep} (EMA) -> {ckpt_path.name}")
        model.load_state_dict(raw_state)

        dt = time.time() - t0
        print(f"  ep {ep:>2d} | tr_loss {tr_loss:.4f} tr_auc {tr_auc:.4f} "
              f"| val[label={held_class}] mean_pred={held_mean:.3f} loss {va_loss:.4f} | {dt:.1f}s")
        history.append({"epoch": ep, "tr_loss": tr_loss, "tr_auc": tr_auc,
                        "va_loss": va_loss, "held_mean": held_mean,
                        "held_class": held_class, "time": dt})

    oof = pd.DataFrame({"Name": vn_canon,
                        "patient_id": val_df["patient_id"].values,
                        "y_true": vy_canon})
    for ep, preds in sorted(snapshot_preds.items()):
        oof[f"y_pred_ep{ep}"] = preds
    oof.to_csv(oof_path, index=False)
    with open(hist_path, "w") as f:
        json.dump({"history": history, "snapshot_epochs": list(snapshot_epochs)},
                  f, indent=2)

    del model, optimizer, sched, scaler, ema, train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return held_class, held_mean

def train_full_data(ckpt_pattern, hist_path, epochs=EPOCHS,
                    seed=SUPERVISED_SEED + 100, snapshot_epochs=SNAPSHOT_EPOCHS):
    """Same training scheme as LOPO (EMA + aux alignment + snapshots), no val
    pass since the full-data model sees all 12 patients."""
    assert "{ep}" in str(ckpt_pattern), "ckpt_pattern must contain '{ep}'"
    seed_everything(seed)
    train_ds = CachedCellDataset(df_train, bf_train_cache, fl_train_cache,
                                 train_modality_transform("bf"),
                                 train_modality_transform("fl"),
                                 paired_tf=PairedGeoAug())
    sampler = PatientBalancedSampler(df_train, batch_size=BATCH_SIZE,
                                     patients_per_batch=PATIENTS_PER_BATCH, seed=seed)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                              worker_init_fn=worker_init_fn)
    model = MultimodalClassifier(pretrained=True, dropout=DROPOUT).to(DEVICE)
    if SSL_ENABLED: load_ssl_branches(model)
    pos = (df_train["Diagnosis"] == 1).sum()
    neg = (df_train["Diagnosis"] == 0).sum()
    pos_weight = torch.tensor(neg / max(pos, 1), device=DEVICE)
    print(f"  full-data: pos_weight={pos_weight.item():.3f}  seed={seed}  epochs={epochs}  "
          f"ls={LABEL_SMOOTHING}  aux={AUX_ALIGN_WEIGHT}  ema={EMA_DECAY}  "
          f"snapshots@{snapshot_epochs}")
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer, max_lrs = make_discriminative_optimizer(model)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=max_lrs,
        steps_per_epoch=len(train_loader), epochs=epochs, pct_start=PCT_START)
    scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None
    ema = ModelEMA(model, decay=EMA_DECAY)
    history = []
    for ep in range(epochs):
        t0 = time.time()
        tr_loss, tr_auc, _, _, _ = run_epoch(
            model, train_loader, optimizer, scaler, criterion, True,
            grad_clip=GRAD_CLIP, sched=sched, log_every=300,
            label_smoothing=LABEL_SMOOTHING,
            aux_align_weight=AUX_ALIGN_WEIGHT, ema=ema,
            sanity_print=(ep == 0))
        dt = time.time() - t0
        print(f"  ep {ep:>2d} | tr_loss {tr_loss:.4f} tr_auc {tr_auc:.4f} | {dt:.1f}s")
        history.append({"epoch": ep, "tr_loss": tr_loss, "tr_auc": tr_auc, "time": dt})
        if ep in snapshot_epochs:
            ckpt_path = Path(str(ckpt_pattern).replace("{ep}", str(ep)))
            torch.save({"model": ema.state_dict(), "epoch": ep,
                        "args": {"dropout": DROPOUT}}, ckpt_path)
            print(f"    [snapshot] saved ep{ep} (EMA) -> {ckpt_path.name}")
    with open(hist_path, "w") as f:
        json.dump({"history": history, "snapshot_epochs": list(snapshot_epochs)},
                  f, indent=2)
    del model, optimizer, sched, scaler, ema, train_loader
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
print("=== Stage 2: LOPO supervised training (multi-snapshot) ===")
splits = leave_one_patient_out_splits(df_train)
print(f"LOPO folds: {len(splits)}   snapshots/fold: {len(SNAPSHOT_EPOCHS)}   "
      f"total LOPO ckpts: {len(splits) * len(SNAPSHOT_EPOCHS)}\n")

for fold, (tr, va) in enumerate(splits):
    held_pat = int(df_train.iloc[va[0]]["patient_id"])
    print(f"=== FOLD {fold} (hold-out pat {held_pat}) ===")
    print("  " + summarize_lopo(df_train, tr, va))
    train_df = df_train.iloc[tr].reset_index(drop=True)
    val_df   = df_train.iloc[va].reset_index(drop=True)
    ckpt_pattern = str(OUT_DIR / f"lopo_pat{held_pat}_ep{{ep}}.pt")
    oof  = OUT_DIR / f"lopo_pat{held_pat}_oof.csv"
    hist = OUT_DIR / f"lopo_pat{held_pat}_history.json"
    expected = [Path(ckpt_pattern.replace("{ep}", str(ep)))
                for ep in SNAPSHOT_EPOCHS]
    if all(c.exists() for c in expected) and oof.exists():
        print(f"  (already trained, all {len(expected)} snapshots present, skipping)\n")
        continue
    train_one_lopo_fold(train_df, val_df, ckpt_pattern, oof, hist)

print("All LOPO folds done.")

In [ ]:
if TRAIN_FULL_DATA_MODEL:
    print("\n=== Stage 3: Full-data model (multi-seed, multi-snapshot) ===")
    for seed in FULL_DATA_SEEDS:
        fd_pattern = str(OUT_DIR / f"fulldata_seed{seed}_ep{{ep}}.pt")
        hist = OUT_DIR / f"fulldata_seed{seed}_history.json"
        expected = [Path(fd_pattern.replace("{ep}", str(ep))) for ep in SNAPSHOT_EPOCHS]
        if all(c.exists() for c in expected):
            print(f"  seed {seed}: already trained, skipping")
            continue
        print(f"\n  --- seed {seed} ---")
        train_full_data(fd_pattern, hist, seed=seed)

In [ ]:
# LOPO has no per-fold val_auc (held-out patient is single-class), so we plot:
#   (1) train BCE loss per fold, (2) train AUC per fold, and (3) the mean predicted
# probability on the held-out patient over epochs, colored by the held-out label.
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
hist_paths = sorted(glob.glob(str(OUT_DIR / "lopo_pat*_history.json")))
for hp in hist_paths:
    h = json.load(open(hp))["history"]
    pat = Path(hp).stem.replace("lopo_pat", "").replace("_history", "")
    cls = h[0].get("held_class", -1)
    color = "tab:red" if cls == 1 else "tab:blue"
    label = f"pat{pat} (y={cls})"
    epochs = [e["epoch"] for e in h]
    ax[0].plot(epochs, [e["tr_loss"] for e in h], marker="o", color=color, alpha=0.7, label=label)
    ax[1].plot(epochs, [e["tr_auc"]  for e in h], marker="o", color=color, alpha=0.7, label=label)
    ax[2].plot(epochs, [e["held_mean"] for e in h], marker="o", color=color, alpha=0.7, label=label)
fd_paths = sorted(glob.glob(str(OUT_DIR / "fulldata_seed*_history.json")))
for j, fd_path in enumerate(fd_paths):
    h = json.load(open(fd_path))["history"]
    seed = Path(fd_path).stem.replace("fulldata_seed", "").replace("_history", "")
    ls = "-" if j == 0 else "--"
    epochs = [e["epoch"] for e in h]
    ax[0].plot(epochs, [e["tr_loss"] for e in h], marker="s", color="black", lw=2, ls=ls,
               label=f"fulldata seed{seed}")
    ax[1].plot(epochs, [e["tr_auc"]  for e in h], marker="s", color="black", lw=2, ls=ls,
               label=f"fulldata seed{seed}")
for a in ax:
    for ep in SNAPSHOT_EPOCHS:
        a.axvline(ep, color="grey", alpha=0.2, lw=1)
ax[0].set(title="Train BCE loss", xlabel="epoch", ylabel="loss")
ax[1].set(title="Train AUC",       xlabel="epoch", ylabel="AUC")
ax[2].set(title="Held-out mean prediction (red=pos, blue=neg)",
          xlabel="epoch", ylabel="mean σ(logit)")
ax[2].axhline(0.5, color="grey", linestyle="--", alpha=0.5)
ax[2].set_ylim(0, 1)
for a in ax: a.legend(fontsize=7, ncol=2); a.grid(True)
plt.tight_layout()
plt.savefig(str(OUT_DIR / "learning_curves.png"), dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Global OOF AUC under multi-snapshot ensembling:
# each cell's prediction is the MEAN of the per-snapshot predictions from the
# fold that held it out. This matches the inference-time ensemble (mean of
# snapshot logits ≈ mean of snapshot sigmoids for the small dynamic range here),
# so the OOF AUC is a faithful proxy for the LB.
oof_paths = sorted(glob.glob(str(OUT_DIR / "lopo_pat*_oof.csv")))
if oof_paths:
    oof = pd.concat([pd.read_csv(p) for p in oof_paths], ignore_index=True)
    snap_cols = sorted(c for c in oof.columns if c.startswith("y_pred_ep"))
    assert snap_cols, "no snapshot prediction columns found in OOF CSVs"
    print(f"OOF rows: {len(oof)}  (should equal train size {len(df_train)})")
    print(f"snapshot columns: {snap_cols}")
    oof["y_pred"] = oof[snap_cols].mean(axis=1)
    cell_auc = roc_auc_score(oof["y_true"], oof["y_pred"])
    # Per-snapshot diagnostic: does averaging actually beat the best single snapshot?
    per_snap_auc = {c: roc_auc_score(oof["y_true"], oof[c]) for c in snap_cols}
    print("\n=== Per-snapshot OOF AUC (cell-level) ===")
    for c, a in per_snap_auc.items():
        print(f"  {c}: {a:.4f}")
    print(f"  mean-of-snapshots: {cell_auc:.4f}")
    pp = oof.groupby("patient_id").agg(
        mean_pred=("y_pred", "mean"), median_pred=("y_pred", "median"),
        n_cells=("Name", "size"),
        label=("y_true", "first")).sort_values("mean_pred")
    pat_auc = roc_auc_score(pp["label"], pp["mean_pred"])
    print("\n=== LOPO OOF (per-patient mean predictions) ===")
    print(pp.to_string())
    print(f"\ncell-level OOF AUC: {cell_auc:.4f}    patient-level AUC: {pat_auc:.4f}")

In [ ]:
def _d4(bf, fl):
    """All 8 D4 transformations (4 rotations × 2 reflections)."""
    for k in range(4):
        bfr = torch.rot90(bf, k, dims=(-2, -1))
        flr = torch.rot90(fl, k, dims=(-2, -1))
        yield bfr, flr
        yield TF.hflip(bfr), TF.hflip(flr)

def load_model_from_ckpt(ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    args = state.get("args", {})
    dropout = float(args.get("dropout", DROPOUT))
    model = MultimodalClassifier(pretrained=False, dropout=dropout).to(DEVICE)
    model.load_state_dict(state["model"], strict=False)
    model.eval()
    return model

def predict_one_ckpt_logits(ckpt_path, loader, tta=True):
    model = load_model_from_ckpt(ckpt_path)
    out_logits = []
    n_aug = 8 if tta else 1
    with torch.no_grad():
        for batch in loader:
            bf = batch["bf"].to(DEVICE, non_blocking=True)
            fl = batch["fl"].to(DEVICE, non_blocking=True)
            l_sum = None
            for bf_t, fl_t in (_d4(bf, fl) if tta else [(bf, fl)]):
                with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
                    li = model(bf_t, fl_t).float()
                l_sum = li if l_sum is None else l_sum + li
            out_logits.append((l_sum / n_aug).cpu().numpy())
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return np.concatenate(out_logits)

test_ds = CachedCellDataset(df_test, bf_test_cache, fl_test_cache,
                            eval_modality_transform("bf"),
                            eval_modality_transform("fl"))
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True,
                         worker_init_fn=worker_init_fn)

ckpts = sorted(glob.glob(str(OUT_DIR / "lopo_pat*_ep*.pt")))
if TRAIN_FULL_DATA_MODEL:
    ckpts += sorted(glob.glob(str(OUT_DIR / "fulldata_seed*_ep*.pt")))
n_lopo = len(glob.glob(str(OUT_DIR / "lopo_pat*_ep*.pt")))
n_full = len(ckpts) - n_lopo
print(f"Ensembling {len(ckpts)} models  "
      f"({n_lopo} LOPO snapshots + {n_full} full-data snapshots):")
for c in ckpts: print("  -", Path(c).name)

all_logits = []
for c in ckpts:
    t0 = time.time()
    all_logits.append(predict_one_ckpt_logits(c, test_loader, tta=True))
    print(f"  {Path(c).name} done in {time.time()-t0:.1f}s")
all_logits = np.stack(all_logits, axis=0)

logit_mean = all_logits.mean(axis=0)
preds_logit = 1.0 / (1.0 + np.exp(-logit_mean))

ranks = np.stack([rankdata(row) for row in all_logits], axis=0)
rank_mean = ranks.mean(axis=0)
preds_rank = (rank_mean - rank_mean.min()) / (rank_mean.max() - rank_mean.min() + 1e-12)

preds = 0.5 * preds_logit + 0.5 * preds_rank

# Write all submission files to Drive (OUT_DIR) so they survive disconnects.
sub_path = OUT_DIR / "submission.csv"
sub = pd.DataFrame({"Name": df_test["Name"].values, "Diagnosis": preds})
sub.to_csv(sub_path, index=False)
print(f"\nWrote {sub_path}  (mean pred = {preds.mean():.3f}, "
      f"min {preds.min():.3f}, max {preds.max():.3f})")

pd.DataFrame({"Name": df_test["Name"].values,
              "Diagnosis": preds_logit}).to_csv(OUT_DIR / "submission_logit.csv", index=False)
pd.DataFrame({"Name": df_test["Name"].values,
              "Diagnosis": preds_rank}).to_csv(OUT_DIR / "submission_rank.csv", index=False)

test_pat_ids = df_test["Name"].map(parse_patient_id).to_numpy()
pat_mean_preds = pd.Series(preds).groupby(test_pat_ids).transform("mean").to_numpy()
pd.DataFrame({"Name": df_test["Name"].values,
              "Diagnosis": pat_mean_preds}).to_csv(OUT_DIR / "submission_patient_agg.csv", index=False)
print("Also wrote submission_logit.csv, submission_rank.csv, "
      "and submission_patient_agg.csv (each cell replaced with its patient's mean).")
print(sub.head())

print(f"\nAll four submissions are in {OUT_DIR}")
print(f"Download `submission.csv` from Drive and upload it to the Kaggle competition page.")
